# Pronóstico de nacimientos en Colombia para 2026

**Investigación aplicada de Juan Diego Londoño Álvarez**

El estudio extiende un pronóstico departamental de 2024 con cifras definitivas de 2024 y 2025. Usa los nacimientos por lugar de ocurrencia en los 32 departamentos y Bogotá, D. C.

Abre el cuaderno desde la raíz del repositorio y usa un entorno con las dependencias de `requirements.txt`. [Fuentes y preparación](data/FUENTES.md)

In [1]:
from pronosticar import cargar_panel, evaluar, estimar, MODELOS, ROOT
import pandas as pd

datos = cargar_panel()
totales = datos.groupby("anio")["nacimientos"].sum()
print(totales.to_string())

anio
2020    629402
2021    616914
2022    573625
2023    515549
2024    453901
2025    441537


## Validación de un año hacia adelante

Los modelos aprenden el cambio respecto al año anterior. Cada prueba usa solo años previos al corte. Se comparan un bosque aleatorio, una regresión lineal y la referencia de repetir el último conteo.

In [2]:
metricas = evaluar(datos)
columnas = ["anio", "modelo", "observado_nacional", "estimado_nacional", "error_nacional_pct", "mae_territorial"]
print(metricas[columnas].round(2).to_string(index=False))
promedios = metricas.groupby("modelo")["mae_territorial"].mean().sort_values()
seleccion = promedios.index[0]
print("\nMAE territorial promedio:")
print(promedios.round(2).to_string())

 anio           modelo  observado_nacional  estimado_nacional  error_nacional_pct  mae_territorial
 2023          ingenuo              515549             573625               11.26          1761.33
 2023 regresion_lineal              515549             460610              -10.66          1675.73
 2023 bosque_aleatorio              515549             533587                3.50           725.21
 2024          ingenuo              453901             515549               13.58          1930.06
 2024 regresion_lineal              453901             410797               -9.50          1312.12
 2024 bosque_aleatorio              453901             461472                1.67           611.97
 2025          ingenuo              441537             453901                2.80           460.85
 2025 regresion_lineal              441537             369494              -16.32          2183.12
 2025 bosque_aleatorio              441537             399030               -9.63          1290.27

MAE terri

## Pronóstico por territorio

Se vuelve a entrenar el modelo seleccionado con todos los años definitivos y se suman las 33 estimaciones enteras para obtener el total nacional.

In [3]:
pronostico = estimar(datos, 2026, MODELOS[seleccion]).sort_values("estimacion", ascending=False)
print("Modelo:", seleccion)
print("Nacimientos estimados:", int(pronostico["estimacion"].sum()))
print(pronostico[["departamento", "nacimientos_previos", "estimacion"]].head(10).to_string(index=False))

Modelo: bosque_aleatorio
Nacimientos estimados: 429879
   departamento  nacimientos_previos  estimacion
         Bogotá                65630       63546
      Antioquia                51916       50789
Valle del Cauca                33856       33074
      Atlántico                26656       25994
        Bolívar                22186       21575
     La Guajira                19966       19701
      Santander                19008       18579
        Córdoba                17087       17568
   Cundinamarca                17789       17332
          Cesar                15878       15504


![Serie nacional observada y pronóstico](resultados/serie_nacional.png)

## Escenario con el corte de 2026

Como comparación, se aplica al total de 2025 el cambio observado entre los cortes de enero a julio de 2025 y 2026. Este supuesto mantiene ese ritmo de variación para el conjunto del año.

In [4]:
corte = pd.read_csv(ROOT / "data/nacimientos_enero_julio.csv").set_index("anio")["nacimientos"]
razon = corte[2026] / corte[2025]
print(f"Cambio en el periodo comparable: {100 * (razon - 1):.2f}%")
print(f"Escenario anual: {round(totales[2025] * razon)} nacimientos")

Cambio en el periodo comparable: 1.51%
Escenario anual: 448222 nacimientos


El bosque reduce el error territorial promedio en los tres cortes de validación, aunque subestima el total de 2025. El cambio en el corte preliminar de 2026 y la diferencia entre estimaciones aconsejan interpretar el resultado como exploratorio.